# Chain-of-Thought: measure both wins from scratch

> Companion notebook to the concept page **17-Chain-of-Thought-Reasoning.md**. Every number printed here is the same number quoted on the page and drawn in the figures — this notebook and `chain_of_thought.py` are one seeded source of truth.

**Chain-of-Thought (CoT)** makes a model emit intermediate **reasoning steps before its answer**. You don't need a real LLM to *see why it works* — you model the *structure* of multi-step reasoning. This notebook builds two deterministic, pure-numpy demonstrations:

1. **Decomposition** — why emitting intermediate steps helps (direct-vs-CoT on a 5-step task).
2. **Self-consistency** — why sampling many chains and majority-voting beats a single chain (Wang et al. 2022).

Each demo **asserts the qualitative result before reporting anything**, so a regression can't hide behind a pretty table.

**Device note:** this demo is pure numpy (CPU) — there are no tensors and no accelerator to use. We still print an honest device line (the detected accelerator, and that we pin numerics to CPU numpy for bit-for-bit reproducibility).

## Setup: honest device line + seed

Pure numpy, so the "device" is CPU. We detect any accelerator only to *report* it honestly, then pin everything to seeded CPU numpy so the numbers are reproducible across machines.

In [1]:
import numpy as np

def detect_device_line():
    """Report the detected accelerator but state we pin numerics to CPU numpy."""
    try:
        import torch  # optional: the demo itself never uses torch
        detected = ("cuda" if torch.cuda.is_available()
                    else "mps" if torch.backends.mps.is_available()
                    else "cpu")
        return (f"device: cpu (detected {detected}; pinned to CPU numpy for reproducibility)"
                f"  |  torch: {torch.__version__}")
    except ModuleNotFoundError:
        return "device: cpu (numpy only; torch not installed)"

print(detect_device_line())
print("numpy:", np.__version__)

device: cpu (detected mps; pinned to CPU numpy for reproducibility)  |  torch: 2.12.0
numpy: 2.4.6


## Demo 1 — decomposition: why emitting intermediate steps helps

We model a **5-step modular-arithmetic chain**: `state_{i+1} = (state_i * a_i + b_i) mod 7`. The final answer is the last state. Two solvers attempt the *same* computation:

- **Direct (blurt the answer):** juggles all five operations at once with no scratch space. Each op survives with a *low* reliability (0.55); one slip with nothing written down derails the rest into a near-random residue. Reliability compounds — roughly `0.55 ** 5`.
- **CoT (show your work):** writes each intermediate result down, turning each step into an *isolated, easier* sub-problem solved with a *higher* reliability (0.90). The emitted intermediate anchors the next step.

The only difference is whether intermediate state is emitted. That asymmetry (`P_STEP_COT > P_STEP_DIRECT`) is the **decomposition effect**: an isolated step with its inputs written down is simply easier than the same step done as part of one all-at-once guess.

> **Step 1 — task + solver constants.** Hoisted so the two solvers and the trial loop share one definition.

In [2]:
N_STEPS, MODULUS = 5, 7
P_STEP_DIRECT, P_STEP_COT = 0.55, 0.90   # isolating a step (writing it down) makes it more reliable
COT_TRIALS, COT_SEED = 20_000, 1

> **Step 2 — the DIRECT solver.** All-or-nothing: correct only if every internal op survives the low per-op reliability; otherwise a slip with no anchor collapses to a near-random residue.

In [3]:
def run_direct(rng, params, x0):
    state, all_ok = x0, True
    for a, b in params:
        state = (state * int(a) + int(b)) % MODULUS   # the correct local update
        if rng.random() >= P_STEP_DIRECT:             # a slip with no scratch space...
            all_ok = False
    return state if all_ok else int(rng.integers(0, MODULUS))  # ...derails into a near-random residue

> **Step 3 — the CoT solver.** Same five steps, but each isolated step is easier (higher reliability) and the emitted intermediate anchors the next step, so a slip is local rather than catastrophic.

In [4]:
def run_cot(rng, params, x0):
    state = x0
    for a, b in params:
        true_next = (state * int(a) + int(b)) % MODULUS    # what this isolated step SHOULD emit
        state = true_next if rng.random() < P_STEP_COT else int(rng.integers(0, MODULUS))
    return state                                           # emit the (maybe wrong) intermediate, continue from it

> **Step 4 — run the trials and assert the qualitative result FIRST.** For each random problem we compute the exact ground truth, then check both solvers. The assert (`CoT > direct`) fires *before* we print anything.

In [5]:
rng = np.random.default_rng(COT_SEED)
direct_hits = cot_hits = 0
for _ in range(COT_TRIALS):
    params = rng.integers(1, MODULUS, size=(N_STEPS, 2))   # (a_i, b_i) per step, in 1..M-1
    x0 = int(rng.integers(0, MODULUS))                     # starting state
    truth = x0
    for a, b in params:                                    # exact ground-truth answer
        truth = (truth * int(a) + int(b)) % MODULUS
    direct_hits += int(run_direct(rng, params, x0) == truth)
    cot_hits    += int(run_cot(rng, params, x0)    == truth)

direct_acc, cot_acc = direct_hits / COT_TRIALS, cot_hits / COT_TRIALS
assert cot_acc > direct_acc, "CoT must beat direct on the compositional task"   # qualitative result FIRST
print(f"direct (one-shot)    = {direct_acc:.3f}")
print(f"CoT (emit each step) = {cot_acc:.3f}")
print(f"gain = +{cot_acc - direct_acc:.3f}   (CoT ~ {cot_acc/direct_acc:.1f}x the direct accuracy)")

direct (one-shot)    = 0.188
CoT (emit each step) = 0.654
gain = +0.466   (CoT ~ 3.5x the direct accuracy)


**Read the gap.** The direct solver lands the right answer ~19% of the time — one slip in five low-reliability steps and it's effectively guessing mod 7. The CoT solver, doing the *same five steps* but each isolated and anchored, hits ~65% — about **3.5×** higher. Nothing got smarter; emitting intermediate state turned one hard joint prediction into five easy local ones. That is the decomposition mechanism, measured.

## Demo 2 — self-consistency: why voting beats a single chain

Now model a noisy reasoner whose single best guess is only **45% reliable** — but whose correct answer is still the single most likely *individual* outcome (its errors scatter uniformly across the other four answers). **Self-consistency** (Wang et al. 2022) samples K chains and takes the **majority (plurality) vote**.

Why does this help so much? Because there are many ways to reach the *correct* answer but each *wrong* path errs in its own scattered way — so as K grows, the vote concentrates on the consistent answer and the scattered errors cancel (a Condorcet-style effect).

> **Step 1 — model constants.** Answer `0` is defined as the correct one; a chain returns `0` with prob `P_CORRECT`, else a uniformly-random wrong answer in `1..N_ANSWERS-1`. Odd K avoids vote ties.

In [6]:
N_ANSWERS, P_CORRECT = 5, 0.45          # one chain: 45% right; the rest spread over 4 wrong answers
K_VALUES = (1, 3, 5, 7, 11, 21, 41)
SC_TRIALS, SC_SEED = 60_000, 0          # 60k trials -> ~0.002 std error, clean curve

> **Step 2 — sample K chains per problem and majority-vote.** A fresh seeded rng per K (`SC_SEED + k`) so the curve isolates the effect of K rather than reflecting a shared random walk.

In [7]:
def self_consistency_sweep():
    acc = {}
    for k in K_VALUES:
        rng = np.random.default_rng(SC_SEED + k)            # fresh stream per K
        correct = rng.random((SC_TRIALS, k)) < P_CORRECT    # True where a chain got it right
        wrong = rng.integers(1, N_ANSWERS, size=(SC_TRIALS, k))
        samples = np.where(correct, 0, wrong)               # correct -> 0, else a random wrong answer
        # majority (plurality) vote per problem; argmax breaks ties toward the lowest index
        voted = np.array([np.bincount(s, minlength=N_ANSWERS).argmax() for s in samples])
        acc[k] = float((voted == 0).mean())                 # fraction where the vote is correct
    return acc

acc = self_consistency_sweep()

> **Step 3 — assert the qualitative results FIRST, then print the rising curve.** Voting must beat a single chain, and accuracy must rise with K (allowing a tiny Monte-Carlo slack between adjacent points).

In [8]:
assert acc[41] > acc[1], "voting must beat a single chain"
assert all(acc[b] >= acc[a] - 0.003 for a, b in zip(K_VALUES, K_VALUES[1:])), "must rise with K"

print(f"{'K':>4} | {'accuracy':>9}")
print("-" * 18)
for k in K_VALUES:
    print(f"{k:>4} | {acc[k]:>8.3f}")
print(f"\nsingle-chain = {acc[1]:.3f}  ->  K=41 vote = {acc[41]:.3f}   (gain +{acc[41]-acc[1]:.3f})")

   K |  accuracy
------------------
   1 |    0.451
   3 |    0.730
   5 |    0.742
   7 |    0.804
  11 |    0.868
  21 |    0.950
  41 |    0.993

single-chain = 0.451  ->  K=41 vote = 0.993   (gain +0.542)


**Read the climb.** A single chain is a coin-flip-ish 0.451. Vote over just 3 chains and you're at 0.730; over 11, 0.868; over 41, **0.993** — near-perfect, from a 45%-reliable base. This is exactly why self-consistency buys large accuracy gains in practice (PaLM-540B GSM8K: 56.6% → 74.4% in Wang et al. 2022) — and why it costs K× the compute.

## Try it yourself

Before running, **predict**: if you raise `P_CORRECT` from 0.45 to 0.49 (still below 0.5), does the K=41 accuracy go *up* or *down*? Now flip it to 0.20 — does voting still climb toward 1, or collapse?

*Hint:* majority vote concentrates on the correct answer **only while it remains the single most likely individual outcome** (the plurality leader). At `P_CORRECT = 0.20` with 4 wrong answers sharing the remaining 0.80 (0.20 each), the correct answer is no longer the unique leader — voting no longer rescues it. The self-consistency guarantee is "the right answer is the *most consistent* one," not "any K fixes any reasoner."

## What we saw

- **Decomposition (Demo 1):** emitting intermediate state turned a ~19% one-shot solver into a ~65% step-by-step solver on the *same* task — one hard joint prediction became five easy local ones. This is why CoT helps multi-step problems.
- **Self-consistency (Demo 2):** majority-voting K diverse chains climbed from 0.45 (single chain) to 0.99 (K=41) — the vote concentrates on the consistent correct answer while scattered wrong answers cancel.
- **Both asserts fired before any number was printed**, and every number here matches the page and the figures (`make_figures_17.py` imports these exact functions).

These two mechanisms — *more serial compute / decomposition* and *sample-and-vote* — are the foundation of everything from self-consistency to tree-of-thoughts to o1-style test-time-compute reasoning models. See the concept page for the full landscape.